# Kubeflow Pipeline

Train the YOLO license-plate model as a pipeline.

**Covered here:** `fetch_data` -> `prepare_data` -> `train`.

References:

- <https://www.kubeflow.org/docs/components/pipelines/getting-started/>
- <https://www.kubeflow.org/docs/components/pipelines/user-guides/core-functions/connect-api/>

## Environment

In [ ]:
# pip install
%pip install -q -U kfp

## Define

- `fetch_data` downloads the DVC-tracked raw dataset from S3.
- `prepare_data` splits it into train/val and writes the ultralytics
  `data.yaml`.
- `train` fine-tunes `yolo11n.pt` and emits `best.pt`.

`dvc_dir_hash` is the md5 in `data/raw.dvc`. It addresses a `.dir` manifest in
S3 listing `{md5, relpath}` for every file, so one hash pins the whole dataset
version.

S3 needs no credentials here: the pod runs as `default-editor`, which carries
the S3 role by EKS Pod Identity (`infra/60-s3-iam.tf`).

Passing an output as the next step's input is what orders the steps; KFP
derives the DAG from the data flow.

In [ ]:
from kfp import dsl
from kfp.dsl import Dataset, Input, Model, Output


@dsl.component(base_image="python:3.12", packages_to_install=["boto3"])
def fetch_data(
    bucket: str,
    dvc_dir_hash: str,
    region: str,
    raw: Output[Dataset],
):
    import json
    from concurrent.futures import ThreadPoolExecutor
    from pathlib import Path

    import boto3

    s3 = boto3.client("s3", region_name=region)

    def dvc_key(md5: str) -> str:
        # DVC shards its content-addressed store by the first two hex chars
        return "dvcstore/files/md5/" + md5[:2] + "/" + md5[2:]

    # the .dir object lists {"md5": ..., "relpath": ...} for every file
    manifest = json.loads(
        s3.get_object(Bucket=bucket, Key=dvc_key(dvc_dir_hash))["Body"].read()
    )

    root = Path(raw.path)
    root.mkdir(parents=True, exist_ok=True)

    def fetch(entry):
        target = root / entry["relpath"]
        target.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(bucket, dvc_key(entry["md5"]), str(target))

    # 1100+ small objects: latency-bound, not bandwidth-bound
    with ThreadPoolExecutor(max_workers=16) as pool:
        list(pool.map(fetch, manifest))

    images = [p for p in root.iterdir()
              if p.suffix.lower() in {".jpeg", ".jpg", ".png"}]
    if not images:
        raise RuntimeError("no images restored -- check the dvc_dir_hash")

    raw.metadata["files"] = len(manifest)
    raw.metadata["images"] = len(images)

    print("fetched", len(manifest), "files /", len(images), "images")


@dsl.component(base_image="python:3.12", packages_to_install=["pyyaml"])
def prepare_data(
    raw: Input[Dataset],
    val_fraction: float,
    split_seed: int,
    processed: Output[Dataset],
):
    import random
    import shutil
    from pathlib import Path

    import yaml

    suffixes = {".jpeg", ".jpg", ".png"}
    src = Path(raw.path)
    dst = Path(processed.path)

    # images and labels pair by basename: foo.jpeg <-> foo.txt
    stems = sorted(p.stem for p in src.iterdir() if p.suffix.lower() in suffixes)
    # seeded, so two runs split the same way and their metrics compare
    random.Random(split_seed).shuffle(stems)
    cut = int(len(stems) * (1 - val_fraction))

    counts = {}
    for split, names in (("train", stems[:cut]), ("val", stems[cut:])):
        for sub in ("images", "labels"):
            (dst / split / sub).mkdir(parents=True, exist_ok=True)
        for stem in names:
            image = next(p for p in src.glob(stem + ".*")
                         if p.suffix.lower() in suffixes)
            shutil.copy(image, dst / split / "images" / image.name)
            label = src / (stem + ".txt")
            # an image with no label file is a legitimate negative sample
            if label.exists():
                shutil.copy(label, dst / split / "labels" / label.name)
        counts[split] = len(names)

    if not counts["train"] or not counts["val"]:
        raise RuntimeError("empty split: " + str(counts))

    class_names = (src / "classes.txt").read_text().split()
    # `path` must be absolute: ultralytics resolves a relative root against the
    # process cwd, then its own DATASETS_DIR, never against this file
    (dst / "data.yaml").write_text(
        yaml.safe_dump(
            {
                "path": str(dst),
                "train": "train/images",
                "val": "val/images",
                "nc": len(class_names),
                "names": class_names,
            },
            sort_keys=False,
        )
    )

    processed.metadata.update(counts)
    print("split", counts, "classes", class_names)


@dsl.component(
    base_image="python:3.12",
    # headless opencv needs no libgl1, which a plain python image lacks;
    # numpy<2 keeps the torch/numpy ABI bridge intact
    packages_to_install=["ultralytics", "opencv-python-headless", "numpy<2"],
)
def train(
    processed: Input[Dataset],
    epochs: int,
    batch: int,
    imgsz: int,
    weights: str,
    model: Output[Model],
):
    import os
    import shutil
    from pathlib import Path

    # the restricted PSS namespace runs an arbitrary UID with no writable HOME
    os.environ["YOLO_CONFIG_DIR"] = "/tmp/ultralytics"
    os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"

    from ultralytics import YOLO

    out = Path(model.path)
    out.mkdir(parents=True, exist_ok=True)

    results = YOLO(weights).train(
        data=str(Path(processed.path) / "data.yaml"),
        epochs=epochs,
        batch=batch,
        imgsz=imgsz,
        device="cpu",
        workers=2,
        project=str(out / "runs"),
        name="train",
        exist_ok=True,
        plots=False,
    )

    # ultralytics names save_dir unpredictably when runs collide, so copy
    # best.pt to a fixed path the next step can rely on
    save_dir = Path(results.save_dir)
    shutil.copy2(save_dir / "weights" / "best.pt", out / "best.pt")

    model.metadata.update({"epochs": epochs, "batch": batch, "imgsz": imgsz})
    print("best.pt written to", out / "best.pt")


@dsl.pipeline
def yolo_pipeline(
    bucket: str = "kubeflow-yolo-dev-099139718958",
    dvc_dir_hash: str = "0e94102a7a6b4424a0f1292c2f221072.dir",
    region: str = "ca-central-1",
    val_fraction: float = 0.2,
    split_seed: int = 0,
    epochs: int = 3,
    batch: int = 8,
    imgsz: int = 640,
    weights: str = "yolo11n.pt",
):
    fetch = fetch_data(bucket=bucket, dvc_dir_hash=dvc_dir_hash, region=region)

    prepare = prepare_data(
        raw=fetch.outputs["raw"],
        val_fraction=val_fraction,
        split_seed=split_seed,
    )

    (
        train(
            processed=prepare.outputs["processed"],
            epochs=epochs,
            batch=batch,
            imgsz=imgsz,
            weights=weights,
        )
        .set_cpu_request("2")
        .set_cpu_limit("4")
        .set_memory_request("6Gi")
        .set_memory_limit("8Gi")
    )

## Compile

Produces a self-contained pipeline yaml. Needs no cluster.

In [ ]:
from kfp import compiler

compiler.Compiler().compile(yolo_pipeline, "yolo_pipeline.yaml")

## Connect

Inside the cluster `kfp.Client()` needs no arguments: it reads the token from
`KF_PIPELINES_SA_TOKEN_PATH` and defaults to
`http://ml-pipeline-ui.kubeflow.svc.cluster.local`.

The token volume comes from a `PodDefault` in this profile namespace.

In [ ]:
import kfp

kfp_client = kfp.Client()

# test the client by listing experiments
experiments = kfp_client.list_experiments(namespace="kubeflow-user-example-com")
print(experiments)

## Run

In [ ]:
run = kfp_client.create_run_from_pipeline_package(
    "yolo_pipeline.yaml",
    arguments={},
)

print(run.run_id)